# Pandas: ściąga do analizy, czyszczenia i przetwarzania danych
Dane: `orders.csv` (308 zamówień, celowo brudne) oraz `customers.csv` (segmenty klientów).

**Problemy ukryte w danych:** duplikaty wierszy, braki (`quantity`, `city`, `payment_method`, `delivery_days`), mieszane formaty dat, ceny jako tekst (`"899,00 zł"`), niespójna wielkość liter i spacje w `city` i `payment_method`, ujemne i odstające ilości, klienci bez dopasowania w `customers.csv`.

In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

## 1. Wczytywanie i zapis

| Funkcja | Definicja | Zastosowanie |
|---|---|---|
| `pd.read_csv()` | Wczytuje plik CSV do DataFrame. | Podstawowy start każdej analizy, z opcjami `sep`, `usecols`, `parse_dates`, `dtype`. |
| `pd.read_excel()` | Wczytuje arkusz Excela. | Gdy dane biznesowe przychodzą w `.xlsx` (parametr `sheet_name`). |
| `df.to_csv()` | Zapisuje DataFrame do CSV. | Eksport oczyszczonych danych; prawie zawsze z `index=False`. |
| `df.copy()` | Tworzy niezależną kopię. | Chroni surowe dane przed modyfikacją i eliminuje `SettingWithCopyWarning`. |

In [2]:
raw = pd.read_csv('orders.csv')
customers = pd.read_csv('customers.csv')
df = raw.copy()

## 2. Inspekcja (pierwszy rzut oka)

| Funkcja | Definicja | Zastosowanie |
|---|---|---|
| `df.head() / df.tail()` | Pokazuje pierwsze / ostatnie n wierszy. | Szybki podgląd struktury i wartości. |
| `df.sample()` | Losuje n wierszy. | Lepszy obraz danych niż `head`, gdy plik jest posortowany. |
| `df.shape` | Krotka (wiersze, kolumny). | Kontrola rozmiaru przed i po czyszczeniu lub łączeniu. |
| `df.info()` | Typy kolumn, liczba wartości niepustych, pamięć. | Wykrywa złe typy (np. cena jako `object`) i braki. |
| `df.dtypes` | Typ każdej kolumny. | Sprawdzenie, czy liczby i daty mają właściwy typ. |
| `df.describe()` | Statystyki opisowe kolumn liczbowych. | Wychwytuje wartości odstające i podejrzane minima (np. ujemna ilość). |
| `s.value_counts()` | Liczność każdej unikalnej wartości. | Ujawnia literówki i niespójne kategorie (`Karta` vs `karta`). |
| `s.unique() / s.nunique()` | Unikalne wartości / ich liczba. | Ocena kardynalności zmiennych kategorycznych. |

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 308 entries, 0 to 307
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        308 non-null    int64  
 1   order_date      308 non-null    str    
 2   customer_id     308 non-null    str    
 3   city            300 non-null    str    
 4   category        308 non-null    str    
 5   product         308 non-null    str    
 6   quantity        295 non-null    float64
 7   unit_price      308 non-null    str    
 8   discount        308 non-null    float64
 9   payment_method  234 non-null    str    
 10  status          308 non-null    str    
 11  delivery_days   303 non-null    float64
dtypes: float64(3), int64(1), str(8)
memory usage: 29.0 KB


In [4]:
df.describe()

,order_id,quantity,discount,delivery_days
count,308.000000,295.000000,308.000000,303.000000
mean,1149.487013,6.389831,0.058442,3.801980
std,85.922527,40.877624,0.073379,1.963338
min,1000.000000,-1.000000,0.000000,1.000000
25%,1075.750000,2.000000,0.000000,2.000000
50%,1149.500000,3.000000,0.050000,4.000000
75%,1222.250000,4.000000,0.100000,6.000000
max,1299.000000,500.000000,0.200000,7.000000


In [5]:
df['city'].value_counts(dropna=False)

city
Kraków      45
wrocław     42
Warszawa    41
 poznań     39
Wrocław     37
KRAKÓW      36
Gdańsk      34
Poznań      26
NaN          8
Name: count, dtype: int64

## 3. Selekcja i filtrowanie

| Funkcja | Definicja | Zastosowanie |
|---|---|---|
| `df['col'] / df[['a','b']]` | Wybiera jedną kolumnę (Series) lub kilka (DataFrame). | Podstawowy wybór kolumn do analizy. |
| `df.loc[]` | Selekcja po etykietach wierszy i nazwach kolumn. | Filtrowanie warunkiem i wybór kolumn jednocześnie: `df.loc[maska, 'col']`. |
| `df.iloc[]` | Selekcja po pozycji (liczby całkowite). | Wycinanie wierszy i kolumn po indeksie pozycyjnym. |
| `maska logiczna` | Series `True/False` użyta jako filtr. | Łączenie warunków przez `&`, `\|`, `~`, zawsze w nawiasach. |
| `df.query()` | Filtrowanie wyrażeniem tekstowym. | Czytelniejsze złożone warunki: `df.query('qty > 2 and status == "delivered"')`. |
| `s.isin()` | Sprawdza przynależność do listy. | Filtr po wielu wartościach kategorii naraz. |
| `s.between()` | Sprawdza zakres wartości (domyślnie domknięty). | Filtrowanie przedziałów cen lub dat. |
| `df.filter()` | Wybiera kolumny po nazwie, `like` lub `regex`. | Szybki wybór grupy kolumn o wspólnym prefiksie. |
| `df.select_dtypes()` | Wybiera kolumny danego typu. | Oddzielenie cech liczbowych od kategorycznych przed modelowaniem. |

In [6]:
df.loc[(df['category'] == 'Elektronika') & (df['quantity'] >= 3), ['order_id', 'product', 'quantity']].head()

,order_id,product,quantity
0,1000,Monitor,4.0
9,1009,Monitor,3.0
18,1018,Słuchawki,3.0
26,1026,Klawiatura,4.0
41,1041,Klawiatura,5.0


In [7]:
df.query("status in ['cancelled', 'returned'] and discount > 0").head()

,order_id,order_date,customer_id,city,category,product,quantity,unit_price,discount,payment_method,status,delivery_days
8,1008,2026-06-01,C035,wrocław,Sport,Hantle,1.0,159,0.05,przelew,cancelled,4.0
10,1010,2026-03-24,C025,Wrocław,Książki,Powieść,2.0,"45,00 zł",0.20,BLIK,returned,3.0
12,1012,2026-02-01,C006,Gdańsk,Dom,Lampa,3.0,129,0.20,przelew,returned,2.0
13,1013,2026-05-29,C027,wrocław,Dom,Poduszka,2.0,59,0.05,BLIK,returned,1.0
20,1020,2026-07-21,C030,wrocław,Dom,Kubek,3.0,29,0.10,BLIK,returned,3.0


## 4. Czyszczenie danych

| Funkcja | Definicja | Zastosowanie |
|---|---|---|
| `df.duplicated()` | Oznacza zduplikowane wiersze. | Policzenie duplikatów przed ich usunięciem. |
| `df.drop_duplicates()` | Usuwa duplikaty (opcjonalnie po `subset`). | Usunięcie powtórzonych zamówień po `order_id`. |
| `df.isna().sum()` | Liczba braków w każdej kolumnie. | Pierwszy krok oceny jakości danych. |
| `df.dropna()` | Usuwa wiersze lub kolumny z brakami. | Gdy braków jest mało lub dotyczą kluczowej zmiennej (`subset`). |
| `df.fillna()` | Uzupełnia braki wartością lub słownikiem. | Imputacja medianą, modą lub stałą (np. `'nieznane'`). |
| `s.str.strip() / .lower() / .title()` | Metody tekstowe na całej kolumnie. | Ujednolicenie zapisu kategorii (`' poznań'` na `'Poznań'`). |
| `s.str.replace()` | Zamiana fragmentów tekstu (też regex). | Usunięcie `zł` i zamiana przecinka na kropkę w cenach. |
| `s.replace()` | Zamiana całych wartości. | Mapowanie wariantów na jedną kategorię. |
| `pd.to_numeric()` | Konwersja na liczby; `errors='coerce'` daje NaN. | Naprawa kolumn liczbowych zapisanych jako tekst. |
| `pd.to_datetime()` | Konwersja na datę; `format='mixed'` dla różnych formatów. | Ujednolicenie dat do analizy czasowej. |
| `s.astype()` | Rzutowanie typu. | Np. `'category'` dla oszczędności pamięci lub `'Int64'` dla liczb z brakami. |
| `s.clip()` | Przycina wartości do zakresu. | Ograniczenie wartości odstających (winsoryzacja). |
| `df.rename()` | Zmienia nazwy kolumn lub indeksu. | Czytelne, spójne nazwy kolumn. |
| `df.drop()` | Usuwa kolumny lub wiersze. | Pozbycie się zbędnych lub błędnych danych. |

In [8]:
print('duplikaty:', df.duplicated().sum())
df = df.drop_duplicates(subset='order_id')
df.isna().sum()

duplikaty: 8


order_id           0
order_date         0
customer_id        0
city               8
category           0
product            0
quantity          12
unit_price         0
discount           0
payment_method    72
status             0
delivery_days      5
dtype: int64

In [9]:
df['city'] = df['city'].str.strip().str.title().fillna('Nieznane')
df['payment_method'] = df['payment_method'].str.lower().fillna('brak')
df['unit_price'] = pd.to_numeric(df['unit_price'].astype(str)
                                 .str.replace(' zł', '').str.replace(',', '.'))
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', dayfirst=True)
df = df[(df['quantity'] > 0) | df['quantity'].isna()]                 # usuń ujemne ilości
df['quantity'] = df['quantity'].fillna(df['quantity'].median()).clip(upper=10)
df['delivery_days'] = df['delivery_days'].fillna(df.groupby('city')['delivery_days'].transform('median'))
df[['category', 'status']] = df[['category', 'status']].astype('category')
df.info()

<class 'pandas.DataFrame'>
Index: 297 entries, 0 to 299
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        297 non-null    int64         
 1   order_date      297 non-null    datetime64[us]
 2   customer_id     297 non-null    str           
 3   city            297 non-null    str           
 4   category        297 non-null    category      
 5   product         297 non-null    str           
 6   quantity        297 non-null    float64       
 7   unit_price      297 non-null    float64       
 8   discount        297 non-null    float64       
 9   payment_method  297 non-null    str           
 10  status          297 non-null    category      
 11  delivery_days   297 non-null    float64       
dtypes: category(2), datetime64[us](1), float64(4), int64(1), str(4)
memory usage: 26.2 KB


## 5. Tworzenie i transformacja kolumn

| Funkcja | Definicja | Zastosowanie |
|---|---|---|
| `df.assign()` | Dodaje nowe kolumny, zwraca nowy DataFrame. | Tworzenie cech w łańcuchu metod (method chaining). |
| `np.where()` | Wektorowy warunek jeśli/w przeciwnym razie. | Szybka zmienna binarna, np. flaga dużego zamówienia. |
| `s.map()` | Mapuje wartości słownikiem lub funkcją. | Kodowanie kategorii albo tłumaczenie etykiet. |
| `df.apply()` | Stosuje funkcję do wierszy lub kolumn. | Logika, której nie da się zwektoryzować (wolniejsze, używać oszczędnie). |
| `pd.cut()` | Dzieli wartości na przedziały o zadanych granicach. | Koszyki cenowe, grupy wiekowe. |
| `pd.qcut()` | Dzieli na przedziały o równej liczności (kwantyle). | Segmentacja klientów na kwartyle wartości. |
| `s.dt.month / .dt.day_name()` | Akcesor dat: rok, miesiąc, dzień tygodnia itd. | Feature engineering z dat. |
| `pd.get_dummies()` | Kodowanie one-hot zmiennych kategorycznych. | Przygotowanie kategorii do modelu ML. |

In [10]:
df = df.assign(
    revenue=lambda d: d['quantity'] * d['unit_price'] * (1 - d['discount']),
    month=lambda d: d['order_date'].dt.to_period('M'),
    weekday=lambda d: d['order_date'].dt.day_name(),
    is_big=lambda d: np.where(d['quantity'] >= 4, 1, 0),
)
df['price_band'] = pd.cut(df['unit_price'], bins=[0, 50, 200, np.inf], labels=['tani', 'średni', 'drogi'])
df[['revenue', 'month', 'weekday', 'is_big', 'price_band']].head()

,revenue,month,weekday,is_big,price_band
0,3416.2,2026-07,Thursday,1,drogi
1,70.2,2026-03,Friday,0,tani
2,159.0,2026-05,Saturday,0,średni
3,636.0,2026-01,Wednesday,1,średni
4,712.0,2026-12,Tuesday,1,średni


## 6. Sortowanie i ranking

| Funkcja | Definicja | Zastosowanie |
|---|---|---|
| `df.sort_values()` | Sortuje po jednej lub kilku kolumnach. | Ranking zamówień; `ascending=[False, True]` dla wielu kluczy. |
| `df.sort_index()` | Sortuje po indeksie. | Porządkowanie wyników `groupby` lub szeregów czasowych. |
| `df.nlargest() / nsmallest()` | Zwraca n największych / najmniejszych wierszy. | Szybkie top N bez pełnego sortowania. |
| `s.rank()` | Nadaje rangi wartościom. | Ranking w grupach razem z `groupby`. |
| `df.set_index() / reset_index()` | Ustawia kolumnę jako indeks / przywraca indeks do kolumny. | `reset_index()` po `groupby` daje płaski DataFrame. |

In [11]:
df.nlargest(5, 'revenue')[['order_id', 'product', 'quantity', 'revenue']]

,order_id,product,quantity,revenue
238,1238,Monitor,5.0,4495.00
266,1266,Monitor,5.0,4495.00
134,1134,Monitor,5.0,4270.25
66,1066,Monitor,4.0,3596.00
193,1193,Monitor,5.0,3596.00


In [12]:
df.sort_values(['category', 'revenue'], ascending=[True, False]).groupby('category', observed=True).head(1)

,order_id,order_date,customer_id,city,category,product,quantity,unit_price,discount,payment_method,status,delivery_days,revenue,month,weekday,is_big,price_band
110,1110,2026-01-27,C035,Gdańsk,Dom,Lampa,5.0,129.0,0.0,brak,delivered,5.0,645.0,2026-01,Tuesday,1,średni
238,1238,2026-01-22,C014,Poznań,Elektronika,Monitor,5.0,899.0,0.0,karta,delivered,6.0,4495.0,2026-01,Thursday,1,drogi
103,1103,2026-11-02,C003,Gdańsk,Książki,Powieść,10.0,45.0,0.0,blik,delivered,1.0,450.0,2026-11,Monday,1,tani
33,1033,2026-02-01,C058,Wrocław,Sport,Hantle,5.0,159.0,0.0,przelew,delivered,6.0,795.0,2026-02,Sunday,1,średni


## 7. Grupowanie i agregacja

| Funkcja | Definicja | Zastosowanie |
|---|---|---|
| `df.groupby()` | Dzieli dane na grupy według kluczy (split, apply, combine). | Podstawa wszystkich podsumowań biznesowych. |
| `.agg()` | Wiele agregacji naraz, także nazwanych. | Raport z sumą, średnią i liczbą w jednej tabeli. |
| `.transform()` | Zwraca wynik grupowy o kształcie oryginału. | Udział w grupie, imputacja medianą grupy. |
| `.size() / .count()` | Liczba wierszy / wartości niepustych w grupie. | Liczność segmentów. |
| `df.pivot_table()` | Tabela przestawna z agregacją. | Przychód: kategorie w wierszach, miesiące w kolumnach. |
| `pd.crosstab()` | Tabela liczności dwóch zmiennych. | Rozkład statusów w kategoriach; `normalize` daje proporcje. |
| `df.resample()` | Grupowanie szeregów czasowych po okresie. | Sprzedaż tygodniowa lub miesięczna (indeks musi być datą). |

In [13]:
(df.groupby('category', observed=True)
   .agg(orders=('order_id', 'count'), revenue=('revenue', 'sum'), avg_basket=('revenue', 'mean'))
   .sort_values('revenue', ascending=False).round(2))

,orders,revenue,avg_basket
category,,,
Elektronika,61,74464.25,1220.73
Sport,85,24218.25,284.92
Dom,88,17688.55,201.01
Książki,63,10041.50,159.39


In [14]:
df['share_in_cat'] = df['revenue'] / df.groupby('category', observed=True)['revenue'].transform('sum')
pd.crosstab(df['category'], df['status'], normalize='index').round(2)

status,cancelled,delivered,returned,shipped
category,,,,
Dom,0.11,0.60,0.15,0.14
Elektronika,0.11,0.64,0.07,0.18
Książki,0.08,0.57,0.14,0.21
Sport,0.13,0.56,0.08,0.22


In [15]:
df.pivot_table(index='category', columns='payment_method', values='revenue', aggfunc='sum', fill_value=0, observed=True).round(0)

payment_method,blik,brak,karta,przelew
category,,,,
Dom,1566.0,3993.0,6708.0,5421.0
Elektronika,12134.0,15898.0,27090.0,19341.0
Książki,2378.0,1700.0,3605.0,2359.0
Sport,4524.0,5342.0,9327.0,5026.0


In [16]:
df.set_index('order_date').resample('ME')['revenue'].sum().round(0).head()

order_date
2026-01-31    21081.0
2026-02-28    10955.0
2026-03-31     9176.0
2026-04-30    14999.0
2026-05-31     8365.0
Freq: ME, Name: revenue, dtype: float64

## 8. Łączenie tabel

| Funkcja | Definicja | Zastosowanie |
|---|---|---|
| `pd.merge() / df.merge()` | Łączenie po kluczu jak SQL JOIN (`how`: inner, left, right, outer). | Dołączenie segmentu klienta do zamówień. |
| `indicator=True` | Dodaje kolumnę `_merge` z pochodzeniem wiersza. | Wykrycie rekordów bez dopasowania. |
| `pd.concat()` | Sklejanie DataFrame'ów pionowo lub poziomo. | Łączenie plików z kilku miesięcy w jeden zbiór. |
| `df.join()` | Łączenie po indeksie. | Szybkie dołączenie tabeli z tym samym indeksem. |

In [17]:
m = df.merge(customers, on='customer_id', how='left', indicator=True)
print(m['_merge'].value_counts())
m.groupby('segment')['revenue'].agg(['count', 'sum']).round(0)

_merge
both          275
left_only      22
right_only      0
Name: count, dtype: int64


,count,sum
segment,,
B2B,88,25805.0
B2C,160,77730.0
VIP,27,15603.0


## 9. Zmiana kształtu danych

| Funkcja | Definicja | Zastosowanie |
|---|---|---|
| `df.melt()` | Format szeroki na długi. | Przygotowanie danych pod seaborn lub do agregacji. |
| `df.pivot()` | Format długi na szeroki (bez agregacji). | Odwrotność `melt`, gdy pary klucz-kolumna są unikalne. |
| `stack() / unstack()` | Przenosi poziom kolumn do indeksu i odwrotnie. | Rozpłaszczenie wyniku `groupby` po dwóch kluczach. |
| `s.explode()` | Rozbija listy w komórkach na osobne wiersze. | Zamówienia z listą produktów w jednej komórce. |

In [18]:
df.groupby(['category', 'status'], observed=True)['revenue'].sum().unstack(fill_value=0).round(0)

status,cancelled,delivered,returned,shipped
category,,,,
Dom,2391.0,10842.0,2389.0,2067.0
Elektronika,3843.0,55906.0,2762.0,11954.0
Książki,841.0,5980.0,1587.0,1634.0
Sport,1943.0,14453.0,1516.0,6307.0


## 10. Szeregi czasowe i okna

| Funkcja | Definicja | Zastosowanie |
|---|---|---|
| `s.shift()` | Przesuwa wartości o n okresów. | Opóźnienia (lag features) do modeli prognostycznych. |
| `s.diff() / pct_change()` | Różnica / zmiana procentowa względem poprzedniego okresu. | Dynamika sprzedaży miesiąc do miesiąca. |
| `s.rolling()` | Okno kroczące (np. średnia z 3 okresów). | Wygładzanie trendu. |
| `s.cumsum()` | Suma skumulowana. | Narastający przychód od początku roku. |

In [19]:
monthly = df.groupby('month')['revenue'].sum().to_frame()
monthly['mom_%'] = monthly['revenue'].pct_change().mul(100).round(1)
monthly['rolling_3'] = monthly['revenue'].rolling(3).mean().round(0)
monthly['ytd'] = monthly['revenue'].cumsum().round(0)
monthly.round(0)

,revenue,mom_%,rolling_3,ytd
month,,,,
2026-01,21081.0,NaN,NaN,21081.0
2026-02,10955.0,-48.0,NaN,32037.0
2026-03,9176.0,-16.0,13738.0,41213.0
2026-04,14999.0,64.0,11710.0,56212.0
2026-05,8365.0,-44.0,10847.0,64577.0
2026-06,6358.0,-24.0,9908.0,70935.0
2026-07,19424.0,206.0,11383.0,90360.0
2026-08,8526.0,-56.0,11436.0,98886.0
2026-09,9653.0,13.0,12535.0,108539.0


## Ćwiczenia (bez podglądania rozwiązań)
1. Który klient (`customer_id`) ma najwyższą łączną wartość zamówień o statusie `delivered`? Podaj top 3.
2. Dla każdej kategorii znajdź produkt z najwyższym przychodem (podpowiedź: `groupby` + `idxmax` lub sortowanie + `head(1)`).
3. Policz odsetek zamówień anulowanych lub zwróconych w każdym mieście. Które miasto wypada najgorzej?
4. Dodaj kolumnę z numerem kolejnego zamówienia danego klienta (podpowiedź: `groupby().cumcount()`).
5. Ile zamówień złożyli klienci, których nie ma w `customers.csv`? Jaki to procent przychodu?
6. Wyznacz średni czas dostawy per dzień tygodnia i zwizualizuj wynik (`.plot(kind='bar')`).

In [20]:
df.to_csv('orders_clean.csv', index=False)